# EPUB → аудиокнига (M4B + EPUB с подсветкой) в Colab

Полный пайплайн репозитория `davnozdu/qwen-epub-audiobook` на бесплатной Tesla T4:

| стадия | команда репозитория | что делает |
|---|---|---|
| 1 | `build_single_voice_manifest.py` (или `prepare_qwen.py run` с ключом) | EPUB → манифест: роли, реплики, главы |
| 2 | `audiobook_epub.py prepare` | нарезка по предложениям + `book-text.epub` для подсветки |
| 3 | `prepare_silero.py` | **ударения**: словарь Supertonic + Silero (как в проде) |
| 4 | `synthesize_qwen_mlx.py` | синтез речи на MLX/CUDA по манифесту |
| 5 | `normalize_book_audio.py` | громкость по LUFS |
| 6 | `mix_book_music.py` | фоновая музыка `resources/Lamplight_and_Paper.mp3` (5 %) |
| 7 | `audiobook_epub.py package` | **`book.m4b` + `book-read-along.epub`** |

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**.
Первый прогон лучше на фрагменте: `TEST_SECONDS = 300` (5 минут звука, ~10–15 мин работы).
Полная книга — часы синтеза, выход держите на Google Drive (есть переключатель ниже).

Голос: образец чтеца (`narrator.wav`, 9,4 с, 24 кГц, моно) **вшит в ноутбук** — загружать ничего не надо.
Прямая речь по требованию самого репозитория читается отдельным голосом `dialog`, но с тем же образцом —
тембр один на всю книгу. `--ephemeral-voices` не используется, поэтому модель VoiceDesign (3 ГБ) не нужна.


## Шаг 1. Окружение

In [ ]:
import os

# Кэш свёрток MLX/CUDA. По умолчанию 128, и генератор падает на середине синтеза:
#   «Cache thrashing is happening, please set the environment variable
#    MLX_CUDA_CONV_CACHE_SIZE to a larger value than 128».
# Это потолок, а не резервация: память берётся по мере надобности, поэтому на T4 с её
# ~15 ГБ ставим 12288 МБ и не экономим (при 8192 трэшинг ещё ловится). Читается первым
# импортом mlx — и в этом ядре,
# и в дочернем процессе синтеза.
os.environ.setdefault("MLX_CUDA_CONV_CACHE_SIZE", "12288")

import platform, shutil, subprocess, sys

print("Python:", sys.version.split()[0])
print("glibc :", ".".join(platform.libc_ver()[1].split(".")[:2]) or "?")
print("ffmpeg:", shutil.which("ffmpeg") or "НЕ НАЙДЕН")

gpu, why = "", []
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=20)
    gpu = out.stdout.strip()
    if not gpu:
        why.append((out.stderr or "nvidia-smi вернул пусто").strip()[:120])
except Exception as e:
    why.append(str(e))

if not gpu:
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            gpu = f"{torch.cuda.get_device_name(0)} ({props.total_memory / 2 ** 30:.0f} ГБ, определён через torch)"
    except Exception as e:
        why.append(f"torch: {e}")

if not gpu:
    try:
        import mlx.core as _mx                    # MLX на Metal тоже даёт gpu
        _device = _mx.default_device()
        if "gpu" in str(_device).lower():
            _info = {}
            try:
                _info = _mx.device_info()
            except Exception:
                pass
            gpu = f"MLX {_info.get('device_name') or _device}"
    except Exception as _exc:
        why.append(f"mlx: {_exc}")

print("GPU   :", gpu or "НЕ НАЙДЕН")
if not gpu:
    print("  почему не нашли:", "; ".join(why) or "неизвестно")
    raise AssertionError(
        "GPU не найден. Runtime → Change runtime type → T4 GPU, затем Runtime → Restart session "
        "и Run all заново (смена рантайма перезапускает машину).")
print("  ориентир: T4 = sm_75, ~15 ГБ доступной памяти")
if "T4" not in gpu:
    print("  это не T4 — не страшно: тест продолжается, цифра будет относиться к вашей карте.")

## Шаг 2. Зависимости

Пины — из `requirements.txt` / `requirements-stress.txt` репозитория, но вместо `mlx-metal` (существует
только под macOS) ставится `mlx[cuda]` — на Linux он подтягивает `mlx-cuda-12` вместе с nvidia-колёсами.
`torch` в образе Colab уже есть, `silero-stress` его использует для ударений.


In [ ]:
%pip install -q "mlx[cuda]==0.32.3" "mlx-audio==0.5.8" "transformers==5.19.0" "huggingface-hub==1.33.0" "soundfile==0.14.0" "EbookLib==0.20" "lxml==6.1.3" "six==1.17.0" "razdel==0.5.0" "beautifulsoup4>=4.12,<5" "silero-stress==1.5"

In [ ]:
import importlib.metadata as md

import mlx.core as mx
import numpy as np

for pkg in ("mlx", "mlx-cuda-12", "mlx-audio", "transformers", "silero-stress", "EbookLib", "soundfile"):
    try:
        print(f"{pkg:16s}", md.version(pkg))
    except Exception:
        print(f"{pkg:16s} НЕ УСТАНОВЛЕН")

device = mx.default_device()
on_gpu = "gpu" in str(device).lower()          # repr выглядит как «Device(gpu, 0)»
print("default device:", device, "| это GPU:", on_gpu)
print("device info   :", mx.device_info())
assert on_gpu, f"MLX выбрал CPU ({device}) — генерация будет бессмысленно медленной"
print("\n✓ GPU и MLX работают")

## Шаг 3. Репозиторий, правки под Linux, словарь ударений

Пять правок, без которых на Linux не поедет: `mlx-metal` → `mlx[cuda]`, пять хардкодов
`/opt/homebrew/bin/ffmpeg` → `/usr/bin`. Плюс одна вещь, которой нет в репозитории: скрипты ударений ждут
словарь в `data/stress/dictionary.json`, а в репозитории он лежит как `resources/stress_dictionary.json`
(8,8 МБ, 100 464 слова с ударением) — копируем на место.


In [ ]:
%cd /content
!rm -rf qwen-epub-audiobook
!git clone -q https://github.com/davnozdu/qwen-epub-audiobook.git
%cd /content/qwen-epub-audiobook

# 1) macOS-only зависимость → CUDA-сборка MLX
!sed -i 's#mlx-metal==0.32.3#mlx[cuda]==0.32.3#' requirements.txt requirements-mlx.txt
# 2) пять хардкодов Homebrew ffmpeg/ffprobe → системный /usr/bin
!sed -i 's#/opt/homebrew/bin/#/usr/bin/#g' audiobook_epub.py normalize_book_audio.py mix_book_music.py synthesize_qwen_mlx.py
# 3) словарь ударений на место, где его ищут скрипты
!mkdir -p data/stress && cp resources/stress_dictionary.json data/stress/dictionary.json && ls -la data/stress

In [ ]:
import pathlib, py_compile, sys

ROOT = pathlib.Path("/content/qwen-epub-audiobook")
sys.path.insert(0, str(ROOT))

IGNORED = {"setup_audiobook.py"}          # маковый установщик, его не запускаем
left = [f"{p.name}:{i}" for p in sorted(ROOT.glob("*.py")) if p.name not in IGNORED
        for i, line in enumerate(p.read_text(encoding="utf-8").splitlines(), 1)
        if "/opt/homebrew/bin" in line]
print("хардкоды Homebrew:", left or "нет")
assert not left

req = (ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines()
mlx_lines = [l for l in req if l.strip().startswith("mlx")]
print("mlx в requirements.txt:", mlx_lines)
assert any("mlx[cuda]" in l for l in mlx_lines) and not any(l.startswith("mlx-metal") for l in mlx_lines)

scripts = ["audiobook_epub.py", "normalize_book_audio.py", "mix_book_music.py", "synthesize_qwen_mlx.py",
           "prepare_qwen.py", "prepare_silero.py", "prepare_speech.py"]
for name in scripts:
    py_compile.compile(str(ROOT / name), doraise=True)
print(f"py_compile: {len(scripts)} скриптов ок")

from stress_dictionary import StressDictionary
dictionary = StressDictionary(str(ROOT / "data/stress"))
print(f"словарь ударений: {len(dictionary.data['corrections'])} слов, sha256 {dictionary.sha256[:12]}…")

from mix_book_music import default_music
print("фоновая музыка:", default_music().name, default_music().stat().st_size // 1024, "КБ")
print("\n✓ репозиторий готов к запуску на Linux")

## Шаг 4. Куда писать результат

Фрагмент (5 минут) спокойно живёт в `/content`. Для полной книги раскомментируйте Drive: сессия живёт
~12 часов, а синтез большой книги идёт дольше, и `progress.json` позволяет продолжить в следующем запуске.


In [ ]:
from pathlib import Path

OUT_ROOT = Path("/content/audiobook")

# Для полной книги лучше Drive (переживает перезапуск рантайма):
# from google.colab import drive; drive.mount("/content/drive")
# OUT_ROOT = Path("/content/drive/MyDrive/audiobook")

PROJECT = OUT_ROOT / "project"          # манифест, роли, библиотека голосов
SOURCE = PROJECT / "source"             # нарезка по предложениям + book-text.epub
AUDIO = PROJECT / "audio"               # сырой синтез + timeline.json + benchmark.json
NORMALIZED = PROJECT / "audio-normalized"
WITH_MUSIC = PROJECT / "audio-music"
RESULT = PROJECT / "result"             # book.m4b + book-read-along.epub
LIBRARY = PROJECT / "voice-library"     # образцы голосов (наш вшитый чтец)
for path in (OUT_ROOT, PROJECT, LIBRARY):
    path.mkdir(parents=True, exist_ok=True)

TEST_SECONDS = 300        # None → вся книга целиком
REBUILD = False           # True → пересобрать подготовку заново (обнулит кэш синтеза)
print("каталог проекта:", PROJECT, "| тестовый фрагмент:", TEST_SECONDS, "с" if TEST_SECONDS else "нет")

## Шаг 5. EPUB

Выберите файл `.epub` — любой, с обложкой и главами. Ниже печатается то, что репозиторий из него извлёк.


In [ ]:
import os
from pathlib import Path

from google.colab import files

uploaded = files.upload()
name = list(uploaded.keys())[0]

# files.upload() кладёт файл в ТЕКУЩИЙ каталог (а мы на шаге 3 перешли в папку репозитория),
# поэтому сначала ищем рядом, потом в /content.
candidates = [Path(name), Path.cwd() / name, Path("/content") / name]
EPUB = next((str(candidate.resolve()) for candidate in candidates if candidate.exists()), "")
print("текущий каталог:", Path.cwd())
assert EPUB, f"файл {name} не найден: искал {[str(c) for c in candidates]}"
print("EPUB:", EPUB, os.path.getsize(EPUB), "байт")

from prepare_qwen import read_source

source = read_source(EPUB)
print("название  :", source["title"])
print("автор     :", source["author"])
print("глав      :", len(source["chapters"]))
print("абзацев   :", len(source["paragraphs"]))
print("знаков    :", sum(len(p["text"]) for p in source["paragraphs"]))

## Шаг 6. Модель синтеза (≈4,6 ГБ, 2–4 мин)

Локальная папка, а не id репозитория: `synthesize_qwen_mlx.py` выставляет `HF_HUB_OFFLINE=1` и работает
только с локальными файлами. Качаем `siriusfreak/qwen3-tts-12hz-1.7b-ru-stress-cf` без `lora/`, `meter/`, `eval/`, `scripts/` —
это лишние 40 МБ чужих скриптов.


In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"      # без xet-реконструкции: обычные чанки, предсказуемо
MODEL_DIR = "/content/models/Qwen3-TTS-12Hz-1.7B-Ru-Stress-CF-source"
!mkdir -p {MODEL_DIR}

from huggingface_hub import snapshot_download

ALLOW = ["config.json", "generation_config.json", "preprocessor_config.json", "tokenizer_config.json",
         "tokenizer.json", "vocab.json", "merges.txt", "model.safetensors", "speech_tokenizer/*"]
path = snapshot_download("siriusfreak/qwen3-tts-12hz-1.7b-ru-stress-cf", local_dir=MODEL_DIR, allow_patterns=ALLOW, max_workers=4)
print("скачано в:", path)

import pathlib

need = ["config.json", "generation_config.json", "preprocessor_config.json", "tokenizer_config.json",
        "vocab.json", "merges.txt", "model.safetensors", "speech_tokenizer/config.json",
        "speech_tokenizer/model.safetensors"]
files = [f for f in pathlib.Path(MODEL_DIR).rglob("*") if f.is_file() and ".cache" not in f.parts]
size = sum(f.stat().st_size for f in files)
missing = [n for n in need if not (pathlib.Path(MODEL_DIR) / n).exists()]
print(f"модель: {size / 1e9:.2f} ГБ | файлов: {len(files)}")
if missing:
    print("чего-то не хватает:", missing)
    !find {MODEL_DIR} -maxdepth 2 -type f | head -30
assert not missing, f"не скачалось: {missing}"
assert size > 4e9, f"веса не докачались: {size / 1e9:.2f} ГБ вместо ~4,5 ГБ"
print("\n✓ модель на месте")

## Шаг 7. EPUB → манифест проекта

Два режима, переключаются автоматически:

- **без ключа** (по умолчанию): `colab/build_single_voice_manifest.py` — весь текст читает один чтец.
  Разметка идёт через собственные проверки репозитория (`validate_cast` / `validate_roles`), поэтому
  целостность текста гарантирована так же, как в облачном режиме. Реплики («— …») выделяются отдельным
  голосом `dialog` — этого требует сам репозиторий, читать прямую речь рассказчиком он запрещает.
- **с ключом** (`OLLAMA_API_KEY` в окружении): `prepare_qwen.py run` — многоголосая разметка LLM,
  как в проде (каждому персонажу своя роль и описание голоса).

Манифест: `project/manifest.json` + `source.json`, `cast.json`, `roles.json`, `segments.jsonl`, `preview.html`.


In [ ]:
import json
import os, subprocess, sys

REPO = "/content/qwen-epub-audiobook"
key = os.environ.get("OLLAMA_API_KEY", "").strip()
if key:
    cmd = [sys.executable, "prepare_qwen.py", "run", EPUB, "-o", str(PROJECT)]
    mode = "многоголосый (LLM Ollama)"
else:
    cmd = [sys.executable, "colab/build_single_voice_manifest.py", EPUB, "-o", str(PROJECT)]
    mode = "один чтец (без ключа)"
print("режим:", mode)

# PYTHONPATH на корень репозитория: скрипт из colab/ импортирует prepare_qwen из корня
env = dict(os.environ, PYTHONPATH=REPO + os.pathsep + os.environ.get("PYTHONPATH", ""))
done = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True, env=env)
print((done.stdout or "")[-1500:])
assert done.returncode == 0, (done.stderr or "")[-2000:]

manifest = json.loads((PROJECT / "manifest.json").read_text(encoding="utf-8"))
print("\nголосов:", len(manifest["voices"]), "| сегментов:", len(manifest["segments"]),
      "| глав:", len(manifest["chapters"]), "| на проверку:", manifest["stats"]["review_segments"])
for voice in manifest["voices"]:
    print(f"  {voice['id']:22s} {voice['name']}")

## Шаг 8. Prepare: нарезка по предложениям и текстовая версия книги

`audiobook_epub.py prepare` превращает манифест в рабочую версию: режет сегменты по предложениям
(паузы 450 мс внутри абзаца, 800 мс между абзацами), оставляет только использованные голоса и собирает
`book-text.epub` — тот самый текст, который потом подсвечивается в плеере.
`--seconds 300` даёт фрагмент; `TEST_SECONDS = None` — вся книга.


In [ ]:
import json

# Повторный запуск не пересобирает book-text.epub: prepare пишет zip с метками времени, хеш
# меняется от запуска к запуску, а он входит в подпись каждого сегмента — пересборка обнулила бы
# весь уже оплаченный GPU-временем кэш синтеза. Совпал фрагмент и стоит REBUILD=False → пропускаем.
prepared_path = SOURCE / "manifest.json"
prepared = json.loads(prepared_path.read_text(encoding="utf-8")) if prepared_path.exists() else None
book_now = json.loads((PROJECT / "manifest.json").read_text(encoding="utf-8"))["book"]["input_sha256"]
if prepared and (SOURCE / "book-text.epub").exists():
    was_seconds = (prepared.get("excerpt") or {}).get("target_seconds") or None
    # origin_book — исходная книга той подготовки; prepare перезаписывает book.input_sha256
    # хешем собранного book-text.epub, поэтому сверяться с ним нельзя
    same_book = (prepared.get("origin_book") or {}).get("input_sha256") == book_now
else:
    was_seconds, prepared, same_book = "нет", None, False

if prepared and same_book and was_seconds == TEST_SECONDS and not REBUILD:
    print("подготовка уже есть — не пересобираю:", prepared_path)
    print(f"  фрагмент тот же: {was_seconds or 'вся книга'}; чтобы пересобрать — REBUILD = True")
else:
    print(f"пересобираю подготовку (книга та же: {same_book}, было: {was_seconds or 'нет'}, "
          f"нужно: {TEST_SECONDS or 'вся книга'}, REBUILD={REBUILD})")
    cmd = [sys.executable, "audiobook_epub.py", "prepare", "--manifest", str(PROJECT / "manifest.json"),
           "--out", str(SOURCE), "--highlight", "sentence", "--start-paragraph", "p00001"]
    if TEST_SECONDS:
        cmd += ["--seconds", str(TEST_SECONDS)]

    done = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
    print((done.stdout or "")[-1200:])
    assert done.returncode == 0, (done.stderr or "")[-2000:]
    prepared = json.loads(prepared_path.read_text(encoding="utf-8"))

print("\nсегментов на озвучку:", prepared["summary"]["segments"],
      "| абзацев:", prepared["summary"]["paragraphs"],
      "| голосов:", prepared["summary"]["voices"])
print("текстовая версия:", (SOURCE / "book-text.epub").name,
      (SOURCE / "book-text.epub").stat().st_size // 1024, "КБ")


## Шаг 9. Ударения — словарь Supertonic + Silero (как в проде)

Это тот же `prepare_silero.py`, что и в `run.command`: сначала словарь (100 464 слова), затем нейросеть
Silero для неизвестных слов и омографов, авторские ударения не перезаписываются. `--llm-disputes off` —
арбитраж спорных случаев облачной LLM выключен (включается `ask`/`run`, если есть ключ).
Результат — ударения `U+0301` после гласной прямо в манифесте.


In [ ]:
import re

cmd = [sys.executable, "prepare_silero.py", "--manifest", str(SOURCE / "manifest.json"),
       "--dictionary-dir", "data/stress", "--llm-disputes", "off"]
done = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
print((done.stdout or "")[-1200:])
assert done.returncode == 0, (done.stderr or "")[-2000:]

prepared = json.loads((SOURCE / "manifest.json").read_text(encoding="utf-8"))
marked = sum(len(re.findall("\u0301", s.get("tts_text", ""))) for s in prepared["segments"])
with_text = sum(1 for s in prepared["segments"] if "\u0301" in s.get("tts_text", ""))
print(f"\nударений в тексте: {marked} | сегментов с ударениями: {with_text} из {len(prepared['segments'])}")
sample = next(s for s in prepared["segments"] if "\u0301" in s.get("tts_text", ""))
print("пример:", sample["tts_text"][:120])
print("движок :", prepared.get("speech_preparation", {}).get("engine", prepared.get("speech", {}).get("engine", "silero+словарь")))

## Шаг 10. Голос чтеца — качается с GitHub

Образец `narrator.wav` (9,4 с, 24 кГц, моно, PCM) лежит в этом же репозитории:
`resources/voices/narrator.wav` + сайдкар `narrator.json` с точной расшифровкой. Ноутбук скачивает его
сам и проверяет sha256, так что вручную загружать ничего не нужно.

Шаг идёт **после** `prepare` намеренно: профиль образца функция `library_profile` связывает с
`book.input_sha256`, а `prepare` перезаписывает это поле хешем собранного `book-text.epub`. Библиотека,
собранная из более раннего манифеста, даёт `ValueError: Описание роли narrator изменилось` уже на синтезе.

Дальше образец кладётся в библиотеку голосов рядом с манифестом: `<voice_id>.wav` + `<voice_id>.json`.
После этого `read_reference` подтверждает образец, и `VoiceDesign` (3 ГБ) для генерации голосов
**не нужен вовсе**. Оба голоса (`narrator` и `dialog`) используют один и тот же файл — книга читается
одним тембром. Хотите свой голос — замените два URL на свои (или положите файл в `/content/narrator.wav`).


In [ ]:
VOICE_URL = "https://raw.githubusercontent.com/davnozdu/qwen-epub-audiobook/main/resources/voices/narrator.wav"
VOICE_META_URL = "https://raw.githubusercontent.com/davnozdu/qwen-epub-audiobook/main/resources/voices/narrator.json"
VOICE_SHA256 = "fffc3115c15ebc1d6a9aac68b776ca99f92cf737e2aa99dcc1d8b8f28b2533ea"

In [ ]:
import hashlib, json, shutil, urllib.request

from synthesize_qwen_mlx import library_profile, read_reference

LIBRARY.mkdir(parents=True, exist_ok=True)
base_wav = LIBRARY / "narrator.wav"
if not base_wav.exists():
    local = Path("/content/narrator.wav")            # сюда можно положить свой файл вместо URL
    if local.exists():
        shutil.copyfile(local, base_wav)
        print("взят локальный файл:", local)
    else:
        urllib.request.urlretrieve(VOICE_URL, base_wav)
        print("образец скачан:", VOICE_URL)

digest = hashlib.sha256(base_wav.read_bytes()).hexdigest()
print("sha256 образца:", digest)
assert digest == VOICE_SHA256, "образец не совпал по sha256 — проверьте VOICE_URL"

try:
    REF_TRANSCRIPT = json.loads(urllib.request.urlopen(VOICE_META_URL, timeout=60).read().decode("utf-8"))["text"]
except Exception as e:
    REF_TRANSCRIPT = ("Усмехнулся тоже и черномазый. Белокурый несколько удивился, что ему удалось "
                      "сказать довольно, впрочем, плохой каламбур.")
    print("сайдкар не скачался, беру расшифровку из ноутбука:", e)
print("расшифровка образца:", REF_TRANSCRIPT[:60], "…")

# Идентичность берём из ПОДГОТОВЛЕННОГО манифеста: prepare перезаписывает book.input_sha256
# хешем собранного book-text.epub, и именно с этим значением синтез сверяет профиль голоса.
manifest = json.loads((SOURCE / "manifest.json").read_text(encoding="utf-8"))
book_hash = manifest["book"]["input_sha256"]
for voice in manifest["voices"]:
    target = LIBRARY / (voice["id"] + ".wav")
    if target != base_wav:
        shutil.copyfile(base_wav, target)             # тот же тембр для реплик
    meta = {"profile": library_profile(voice, book_hash), "speaker": voice["id"], "name": voice["name"],
            "ref_text": REF_TRANSCRIPT, "source": "resources/voices/narrator.wav (9,4 с)",
            "audio_seconds": 9.4, "sample_rate": 24000,
            "wav_sha256": hashlib.sha256(target.read_bytes()).hexdigest()}
    (LIBRARY / (voice["id"] + ".json")).write_text(json.dumps(meta, ensure_ascii=False), encoding="utf-8")
    ref = read_reference(LIBRARY, voice, book_hash)
    print(f"{voice['id']:22s} образец принят: {len(ref['ref_text'])} знаков текста → {Path(ref['ref_audio']).name}")
print("\n✓ голоса готовы, VoiceDesign не понадобится")

## Шаг 11. Синтез речи (MLX/CUDA)

Сначала `--plan` — проверка выбора и валидация без загрузки модели: печатает, сколько сегментов и ролей
пойдёт в работу. Затем реальный прогон.

- `--mode clone --rab-generation` — рабочая схема репозитория: образец голоса как waveform, нативные
  настройки сэмплера (`temperature 0.9`);
- `--stress-marks keep` — ударения из манифеста уходят в модель;
- `--voice-library` — наша библиотека с вшитым образцом, поэтому `--design-model` не нужен;
- `--whole-book` стоит всегда: длину фрагмента задал `prepare` на шаге 8, а синтез обязан
  озвучить весь этот манифест — иначе `timeline.json` короче манифеста и упаковка падает;
- прогресс идёт по сегментам, `timeline.json` и `benchmark.json` пишутся в `project/audio`,
  повторный запуск продолжает с места остановки (по кэшу сегментов).


In [ ]:
BASE = [sys.executable, "synthesize_qwen_mlx.py", "--manifest", str(SOURCE / "manifest.json"),
        "--model", MODEL_DIR, "--out", str(AUDIO), "--mode", "clone", "--rab-generation",
        "--stress-marks", "keep", "--temperature", "0.9", "--voice-library", str(LIBRARY),
        "--start-paragraph", "p00001"]
# --whole-book обязателен всегда: границу фрагмента задаёт prepare (--seconds).
# Без него синтез останавливается по своей оценке длительности, в timeline окажется
# меньше сегментов, чем в манифесте, и упаковка упадёт: «Не озвучена вся книга или
# изменён порядок фрагментов».
BASE.append("--whole-book")

done = subprocess.run(BASE + ["--plan"], cwd=REPO, text=True, capture_output=True)
print("ПЛАН:", (done.stdout or "")[-800:])
assert done.returncode == 0, (done.stderr or "")[-1500:]

In [ ]:
import os, time

CACHE_STEPS = [int(os.environ.get("MLX_CUDA_CONV_CACHE_SIZE", "12288")), 16384]


def synth_once(cache_mb):
    """Прогон синтеза с явно заданным потолком кэша свёрток; возвращает (код, секунды, вывод)."""
    env = dict(os.environ, MLX_CUDA_CONV_CACHE_SIZE=str(cache_mb))
    started = time.time()
    process = subprocess.Popen(BASE, cwd=REPO, text=True, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, bufsize=1, env=env)
    tail = []
    for line in process.stdout:
        print(line.rstrip(), flush=True)
        tail.append(line)
        del tail[:-60]
    process.wait()
    return process.returncode, time.time() - started, "".join(tail)


# «Cache thrashing» — единственная ошибка, которую лечит повтор с другим потолком кэша.
# Готовые сегменты лежат в кэше, так что повтор продолжает с места остановки.
for cache_mb in CACHE_STEPS:
    code, seconds, log = synth_once(cache_mb)
    if code == 0:
        print(f"\nсинтез занял {seconds:.0f} с; потолок кэша свёрток MLX/CUDA — {cache_mb} МБ")
        break
    if "Cache thrashing" in log and cache_mb != CACHE_STEPS[-1]:
        print(f"\nсинтез упал на потолке {cache_mb} МБ (Cache thrashing) — повторяю с "
              f"{CACHE_STEPS[CACHE_STEPS.index(cache_mb) + 1]} МБ", flush=True)
        continue
    raise AssertionError(f"синтез упал с кодом {code} (потолок кэша свёрток {cache_mb} МБ)")

import statistics

report = json.loads((AUDIO / "benchmark.json").read_text(encoding="utf-8"))
rows = report["segments"]
print("состояние:", report["state"], "| звука:", round(report["final_audio_seconds"], 1), "с",
      "| синтез:", round(report["synthesis_seconds"], 1), "с")
print(f"RTF {report['rtf']:.2f} ({report['realtime_speed']:.2f}× realtime) | загрузка модели "
      f"{report['model_load_seconds']:.1f} с | сегментов {len(rows)}, из кэша {report['cached_segments']}")

# Первый вызов включает прогрев (на CUDA это десятки секунд) — без него цифра честнее.
if len(rows) > 1:
    rest = rows[1:]
    audio = sum(r["audio_seconds"] for r in rest)
    spent = sum(r["synthesis_seconds"] for r in rest)
    print(f"без первого сегмента (прогрев): RTF {spent / audio:.2f} ({audio / spent:.2f}× realtime)")
    print(f"первый сегмент: {rows[0]['audio_seconds']:.1f} с звука за {rows[0]['synthesis_seconds']:.1f} с")

# Короткие сегменты дороже: у каждого вызова есть постоянные накладные.
for label, low, high in (("до 2 с", 0, 2), ("2–6 с", 2, 6), ("больше 6 с", 6, 10 ** 9)):
    group = [r for r in rows if low < r["audio_seconds"] <= high]
    if group:
        print(f"  {label:11} n={len(group):4} | средняя длина "
              f"{statistics.mean(r['audio_seconds'] for r in group):5.1f} с | "
              f"RTF {statistics.mean(r['rtf'] for r in group):5.2f}")
print("  накладные на вызов делятся на длину сегмента — на коротком тесте RTF всегда хуже")

## Шаг 12. Громкость (LUFS) и фоновая музыка

In [ ]:
done = subprocess.run([sys.executable, "normalize_book_audio.py", "--audio-dir", str(AUDIO), "--out", str(NORMALIZED)],
                      cwd=REPO, text=True, capture_output=True)
print((done.stdout or "")[-600:])
assert done.returncode == 0, (done.stderr or "")[-1200:]

In [ ]:
done = subprocess.run([sys.executable, "mix_book_music.py", "--audio-dir", str(NORMALIZED), "--out", str(WITH_MUSIC),
                       "--volume-percent", "5"], cwd=REPO, text=True, capture_output=True)
print((done.stdout or "")[-600:])
assert done.returncode == 0, (done.stderr or "")[-1200:]
print("\nфайлы со музыкой:", sorted(p.name for p in WITH_MUSIC.glob("*"))[:6])

## Шаг 13. M4B + EPUB с подсветкой

`audiobook_epub.py package` собирает M4B (EbookLib + FFmpeg) и version-EPUB с media overlays — в плеере
строка подсвечивается синхронно со звуком. Рядом кладётся `book-read-along.epub`.


In [ ]:
done = subprocess.run([sys.executable, "audiobook_epub.py", "package", "--manifest", str(SOURCE / "manifest.json"),
                       "--audio-dir", str(WITH_MUSIC), "--out", str(RESULT)],
                      cwd=REPO, text=True, capture_output=True)
print((done.stdout or "")[-800:])
assert done.returncode == 0, (done.stderr or "")[-1500:]

for path in sorted(RESULT.glob("*")):
    print(f"  {path.name:28s} {path.stat().st_size / 1e6:8.1f} МБ")

In [ ]:
from google.colab import files

for name in ("book.m4b", "book-read-along.epub"):
    path = RESULT / name
    if path.exists():
        files.download(str(path))
    else:
        print("нет файла:", path)

## Тайминги, продолжение и подводные камни

**Что сколько занимает** (T4, фрагмент 5 минут звука): установка ~1 мин, модель 4,6 ГБ ~2–4 мин,
манифест секунды, ударения 1–3 мин (Silero на CPU), синтез — зависит от RTF, сведение и упаковка минуты.
Полная книга (10 ч звука) при RTF 0,5 — около 5 часов, то есть одна сессия Colab целиком, при RTF 1,0 —
две сессии с продолжением.

**Продолжение после обрыва:** вынести `OUT_ROOT` на Google Drive и запускать те же ячейки снова —
`progress.json` в каталоге аудио и кэш сегментов не дают пересинтезировать готовое. Модель в `/content`
не переживает перезапуск рантайма (скачается заново в новом сеансе).

**Многоголосый режим:** задайте `OLLAMA_API_KEY` (`os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")`),
и шаг 7 пойдёт через `prepare_qwen.py run`: LLM найдёт персонажей, каждому даст свою роль и описание
голоса. Тогда для новых голосов понадобится `--design-model` (VoiceDesign 8-bit, 3 ГБ) — добавьте в шаг 11
`--design-model /content/models/Qwen3-TTS-12Hz-1.7B-VoiceDesign-8bit` и скачайте его в шаге 6
(`hf download mlx-community/Qwen3-TTS-12Hz-1.7B-VoiceDesign-8bit --local-dir ...`), а `--voice-library`
оставьте — вшитый образец подхватится для того голоса, чей профиль совпадёт.

**Если что-то не так:**

- `ImportError: libnvrtc.so.12` → `%pip install -q nvidia-cuda-nvrtc-cu12 nvidia-cublas-cu12`;
- `Не импортированы словари Supertonic` → не выполнен шаг 3 (`data/stress/dictionary.json`);
- `Образец без текста` / `Опубликованный профиль не совпадает` → шаг 8 перезапустить после шага 9,
  порядок именно такой: манифест → библиотека голосов → prepare → ударения → синтез;
- `Ollama HTTP 401` → ключ в окружении пустой или протух, нужен только для многоголосого режима;
- звук с треском или тишиной → MLX на вашей карте, скорее всего, использует bf16-путь; проверьте
  torch-вариант из `qwen_tts_colab_smoke_torch.ipynb` на том же образце.

**Не запускать:** `setup_audiobook.py` (маковый установщик), `build_audiobook.py` (интерактивный),
`*.command` (зовут `/opt/homebrew/opt/python@3.11`).

## Почему на коротком тесте скорость хуже

RTF зависит от длины сегмента: у каждого вызова синтеза есть постоянные накладные, а первый вызов
включает прогрев (на T4 это ~70 с). Замер по сегментам одного прогона: до 2 с — RTF ≈ 1,4; 2–6 с —
≈ 1,1; длиннее 6 с — ≈ 0,9. На двухминутной книге этих накладных больше, чем полезной работы, поэтому
цифра пессимистичная; судить о T4 нужно по 5-минутному фрагменту или по книге целиком.

Если нужен быстрее и не жалко подсветки: `prepare --highlight fragment` не режет текст по предложениям,
сегментов в разы меньше, а значит меньше накладных на вызовы (подсветка в плеере станет поабзацной).
